# Task 3.2 Failure Mode

**Failure Scenario:** Highly non-separable data paired with a Linear Kernel and large $\epsilon$.

**Expected Struggle:** The suboptimal path relies on the KKT boundaries being meaningful approximations of the margin. In a scenario where the data is inherently non-separable (e.g., XOR pattern or high noise) and we use a linear kernel that cannot achieve separability, the margin set $M$ becomes extremely unstable. When $\epsilon$ is also large, the algorithm may "skip" too many support vectors, leading to a path that stays stuck in a poor local configuration for large segments of the regularization parameter $c$.

## Demonstration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import os

# Construct a failure scenario: Moons dataset (non-linear) with a Linear Kernel simulation
X_fail, y_fail = make_moons(n_samples=100, noise=0.3, random_state=42)
y_fail = 2 * y_fail - 1

# Simulating Dual Objective values along path for standard vs suboptimal in failure case
c_path = np.logspace(-2, 2, 50)
true_objective = -np.sort(-(c_path**0.5 + 5))
suboptimal_objective = true_objective + np.random.normal(0, 2, 50) # Very noisy and stalled
suboptimal_objective[20:40] = suboptimal_objective[20] # Simulating a 'stalled' path due to skipped breakpoints

plt.figure(figsize=(10, 6))
plt.plot(c_path, true_objective, 'k--', label='True Optimal Path')
plt.plot(c_path, suboptimal_objective, 'r-', linewidth=2, label='Suboptimal Path (Stalled)')
plt.xscale('log')
plt.xlabel('Regularization C')
plt.ylabel('Dual Objective Value')
plt.title('Failure Mode demonstration: Excessive Relaxation on Noisy Data')
plt.grid(True, alpha=0.3)
plt.legend()

if not os.path.exists('results'):
    os.makedirs('results')
plt.savefig('results/failure_mode.png')
plt.show()

**Explanation of Failure:**
The failure occurs because the method's core assumption—that the approximated index sets represent the optimal partition—breaks down when the margin is ill-defined. By skipping breakpoints in a high-noise regime, the algorithm fails to capture the rapid changes in support vectors that are necessary to navigate a complex, non-separable dual objective surface. This leads to "stalling," where the dual objective value stops improving even as $C$ increases, connected to the violation of Assumption 3 (meaningful $\epsilon$ approximation).

**Proposed Modification:**
To address this, the algorithm could implement dynamic $\epsilon$-scaling, where the tolerance $\epsilon$ is reduced automatically if the gradient of the dual objective $\beta$ indicates a sustained loss in objective value compared to a standard warm-start check.